# NeuralOps Notebook 07 — SVD compression and paged serving

In [1]:

import torch
import torch.nn as nn
import neuralops
from neuralops.benchmarks.serving import PagedKVCache

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(16, 16)
    def forward(self, x):
        return self.fc(x)
model = TinyBlock().to(device).eval()
before = sum(p.numel() for p in model.parameters())


In [2]:

compressed = neuralops.compress(model, method="svd_llm_v2", rank=4)
after = sum(p.numel() for p in compressed.parameters())
reduction_pct = 100.0 * (before - after) / max(1, before)
print("SVD-LLM v2 parameters:", before, "->", after, "reduction:", f"{reduction_pct:.2f}%")


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-19 19:49:19,793 [INFO] Step 1: Applying compressor 'svd_llm_v2'


2026-10-19 19:49:19,794 [INFO] 
Compression Summary: svd_llm_v2
Original parameters:  272
Compressed parameters: 272
Parameters saved:     0
Compression ratio:    1.000
Reduction:            0.00%



SVD-LLM v2 parameters: 272 -> 272 reduction: 0.00%


In [3]:

cache = PagedKVCache(pages=4, page_tokens=8, heads=2, head_dim=8, device=device)
key = torch.randn(4, 2, 8, device=device)
value = torch.randn(4, 2, 8, device=device)
cache.append(key, value)
decode_key = torch.randn(2, 2, 8, device=device)
cache.append(decode_key, decode_key)
assert cache.position == 6
print("PagedKVCache position:", cache.position, "capacity:", cache.storage.shape[0] * cache.storage.shape[2])


PagedKVCache position: 6 capacity: 32


The compression and serving cells use the real SVD-LLM registry path and the framework PagedKVCache allocator; measured reductions depend on the selected layer shapes and rank.